In [ ]:
"""
workflows.py
============

A WORKFLOW is just Claude calls arranged in an order that YOU decide in code.
This file shows the five classic patterns, each as its own small function you
can read from top to bottom. They are all wired to one running example: a
customer-support assistant.

The five patterns:
    1. Prompt chaining      - do a task in fixed steps, one after another
    2. Routing              - sort the request, then send it to a specialist
    3. Parallelization      - run several calls at once, then combine them
    4. Orchestrator-workers - let Claude split a task, do the pieces, merge them
    5. Evaluator-optimizer  - draft, grade, improve (a limited number of times)

Also included:
    - dispatch(): a top-level router that picks WHICH workflow to run
    - demo():     runs every pattern once and prints the result

How to run it:
    1. pip install anthropic
    2. export ANTHROPIC_API_KEY=sk-ant-your-key-here
    3. python workflows.py
"""

# "os" lets us read the API key from the environment (we don't hardcode keys).
import os

# ThreadPoolExecutor lets us run a few Claude calls at the SAME time.
# We only use it in the parallelization pattern (pattern #3).
from concurrent.futures import ThreadPoolExecutor

# This is the official Anthropic library that talks to Claude.
from anthropic import Anthropic


# ===========================================================================
# WORKFLOW vs AGENT  (this is just a note. It doesn't run.)
# ===========================================================================
#
# Use a WORKFLOW (like everything in this file) when you already know the steps
# the task needs. Workflows are predictable, cheaper, faster, and easy to debug
# because YOU wrote the path in code.
#
# Use an AGENT only when the task is truly open-ended -- when you can't list the
# steps ahead of time and Claude has to decide what to do next on its own.
# Agents are more flexible but slower, pricier, and harder to control, so they
# need guardrails (like a limit on how many steps they may take).
#
# Simple rule: start with the simplest thing that works. Reach for an agent only
# when a workflow genuinely can't handle the job.
WORKFLOW_VS_AGENT = "See the note above: workflows for known steps, agents for open-ended tasks."


# ===========================================================================
# THE SHARED DOORWAY TO CLAUDE
# ===========================================================================

# Create ONE client for the whole program. It automatically reads your key from
# the ANTHROPIC_API_KEY environment variable.
client = Anthropic()

# The model we'll use for every call. Keeping it in one place means you only
# have to change this one line to switch models.
MODEL = "claude-sonnet-5"


def call_claude(system, user):
    """Send one message to Claude and return its reply as plain text.

    Every pattern in this file calls Claude through THIS function. Writing it
    once means we don't repeat ourselves, and if anything ever needs fixing,
    we fix it here.

    system = the instructions / role for Claude (e.g. "You are a billing agent")
    user   = the actual thing we want Claude to work on
    """
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,          # the longest reply we'll allow
        system=system,
        messages=[{"role": "user", "content": user}],
    )

    # Claude's reply can come back in several "blocks". Sometimes the first block
    # is a hidden "thinking" block, not the answer. So we look through the blocks
    # and return the first one that is actual text.
    for block in response.content:
        if block.type == "text":
            return block.text.strip()

    # If no text came back for some reason, return an empty string.
    return ""


# ===========================================================================
# PATTERN 1: PROMPT CHAINING
# Do a task in a fixed order of steps. Each step's answer feeds the next one.
# ===========================================================================

def prompt_chaining(message):
    """Handle a support message in two clear steps, plus a safety 'gate'.

    Step 1: summarize what the customer actually needs.
    Gate:   if there's no real issue, stop here (don't waste step 2).
    Step 2: write a friendly reply based on that summary.
    """
    # --- Step 1: figure out the core issue -----------------------------
    summary = call_claude(
        "Summarize the customer's main issue in one sentence. "
        "If there is no real issue, reply with exactly: NONE",
        message,
    )

    # --- Gate: a simple quality check before we continue ---------------
    # If step 1 said "NONE", the message wasn't a real request, so we stop.
    if "NONE" in summary.upper():
        return "No clear issue found, so the chain stopped early."

    # --- Step 2: write the reply using the summary from step 1 ---------
    reply = call_claude(
        "Write a short, friendly support reply that solves this issue.",
        summary,
    )
    return reply


# ===========================================================================
# PATTERN 2: ROUTING
# First decide what KIND of request it is, then send it to the right specialist.
# ===========================================================================

def routing(message):
    """Sort a message into billing vs. technical, then use a focused prompt."""
    # First, ask Claude to label the message with a single category.
    category = call_claude(
        "Classify this support message. Reply with ONE word only: "
        "'billing' or 'technical'.",
        message,
    ).lower()

    # Then send it to the matching specialist. Plain if/elif -- easy to follow.
    if "billing" in category:
        reply = call_claude(
            "You are a billing specialist. Help with this charge or invoice.",
            message,
        )
    else:  # anything that isn't billing we treat as technical
        reply = call_claude(
            "You are a technical support agent. Give clear troubleshooting steps.",
            message,
        )

    # Return both the label and the reply so we can see which branch ran.
    return category, reply


# ===========================================================================
# PATTERN 3: PARALLELIZATION
# Run several INDEPENDENT calls at the same time, then combine the answers.
# ===========================================================================

def parallelization(message):
    """Pull three separate things out of one ticket, all at once.

    Checking for a billing issue, a technical issue, and positive feedback are
    independent jobs, so we run them together to save time.
    """
    # A small helper that asks Claude to extract one specific thing.
    def extract(what):
        return call_claude(
            "Extract the " + what + " from this message in one short line. "
            "If there is none, say 'none'.",
            message,
        )

    # Start all three calls at the SAME time using a thread pool.
    with ThreadPoolExecutor() as pool:
        billing_job = pool.submit(extract, "billing problem")
        technical_job = pool.submit(extract, "technical problem")
        praise_job = pool.submit(extract, "positive feedback")

        # .result() waits for each call to finish and gives us its answer.
        billing = billing_job.result()
        technical = technical_job.result()
        praise = praise_job.result()

    # Combine the three answers into one summary.
    return (
        "Billing:   " + billing + "\n"
        "Technical: " + technical + "\n"
        "Praise:    " + praise
    )


# ===========================================================================
# PATTERN 4: ORCHESTRATOR-WORKERS
# Let Claude split a task into pieces, do each piece, then merge the results.
# (Like parallelization, but Claude decides the pieces instead of us.)
# ===========================================================================

def orchestrator_workers(task):
    """Split a task at runtime, do each part, then combine into one answer."""
    # --- Orchestrator: ask Claude to break the task into a few subtasks ---
    plan = call_claude(
        "Break this task into 2 or 3 small subtasks. "
        "Put each subtask on its own line, with nothing else.",
        task,
    )

    # Turn the plan (one subtask per line) into a clean list of subtasks.
    subtasks = []
    for line in plan.splitlines():
        line = line.strip("-* ").strip()   # remove bullets and spaces
        if line:                            # skip blank lines
            subtasks.append(line)

    # --- Workers: do each subtask (one call per subtask) ----------------
    worker_results = []
    for subtask in subtasks:
        result = call_claude(
            "You are a worker. Complete this one subtask briefly.",
            subtask,
        )
        worker_results.append("Subtask: " + subtask + "\nResult: " + result)

    # --- Synthesizer: merge all the worker results into one answer ------
    combined = "\n\n".join(worker_results)
    final = call_claude(
        "Combine these results into one clear, final answer.",
        combined,
    )
    return final


# ===========================================================================
# PATTERN 5: EVALUATOR-OPTIMIZER
# Draft, grade the draft, improve it -- but only a LIMITED number of times.
# ===========================================================================

def evaluator_optimizer(task, max_rounds=3):
    """Write a draft, have a separate 'grader' check it, and improve it.

    The most important part is 'max_rounds': the loop can run at most this many
    times. That limit is what keeps this a safe workflow instead of a loop that
    could run forever.
    """
    # The clear standards the grader will check against.
    criteria = "Warm, takes responsibility, offers a next step, and is 2 sentences or fewer."

    # --- Write the first draft -----------------------------------------
    draft = call_claude("You are a helpful writer. Do the task.", task)

    # Try to improve the draft, up to max_rounds times.
    round_number = 1
    while round_number <= max_rounds:
        # --- A SEPARATE grader checks the draft against the criteria ----
        grade = call_claude(
            "You are a strict grader. Check the draft against these rules:\n"
            + criteria +
            "\nReply with 'PASS' or 'FAIL' on the first line, then say why.",
            "TASK: " + task + "\n\nDRAFT: " + draft,
        )

        # If the grader is happy, we're done -- stop early.
        if grade.upper().startswith("PASS"):
            print("    (passed on round " + str(round_number) + ")")
            break

        # Otherwise, rewrite the draft using the grader's feedback.
        draft = call_claude(
            "Improve the draft using this feedback. Return only the new draft.",
            "DRAFT: " + draft + "\n\nFEEDBACK: " + grade,
        )
        round_number = round_number + 1

    return draft


# ===========================================================================
# TOP-LEVEL ROUTER
# Look at a request and pick WHICH of the workflows above to run.
# ===========================================================================

def dispatch(request):
    """Choose the right workflow for a request, then run it."""
    # Ask Claude which kind of handling this request needs.
    kind = call_claude(
        "Pick the best way to handle this request. Reply with ONE word only:\n"
        "'simple'  = a quick direct answer\n"
        "'research'= needs to be broken into parts\n"
        "'quality' = needs careful, polished writing",
        request,
    ).lower()

    print("  [router picked: " + kind + "]")

    # Run the matching workflow. Plain if/elif so it's easy to follow.
    if "research" in kind:
        return orchestrator_workers(request)
    elif "quality" in kind:
        return evaluator_optimizer(request)
    else:  # "simple" or anything unexpected -> just answer directly
        return call_claude("Answer the request directly and briefly.", request)


# ===========================================================================
# DEMO: run each pattern once so you can SEE them work.
# ===========================================================================

def demo():
    print("PATTERN 1 - PROMPT CHAINING")
    print(prompt_chaining("I was charged twice this month and want a refund."))
    print()

    print("PATTERN 2 - ROUTING")
    category, reply = routing("The app crashes when I open settings.")
    print("routed to:", category)
    print(reply)
    print()

    print("PATTERN 3 - PARALLELIZATION")
    print(parallelization(
        "I was charged twice, the app is slow, but I love the dark mode!"
    ))
    print()

    print("PATTERN 4 - ORCHESTRATOR-WORKERS")
    print(orchestrator_workers(
        "Write a short onboarding checklist for a new support hire."
    ))
    print()

    print("PATTERN 5 - EVALUATOR-OPTIMIZER")
    print(evaluator_optimizer(
        "Write an apology to a customer whose order arrived broken."
    ))
    print()

    print("TOP-LEVEL ROUTER")
    print(dispatch("What are your support hours?"))


# This line means: only run demo() when you launch this file directly
# (python workflows.py), not when another file imports it.
if __name__ == "__main__":
    demo()
